# Stock Volatility Trend Analysis

An exploratory study of price trends, return distributions, volatility clustering, drawdowns, and regime identification for a simulated mid-cap equity over five years.

## Project Overview

This notebook simulates five years of daily OHLCV (Open, High, Low, Close, Volume) data for a mid-cap stock, including a growth regime, a crash period, and a recovery phase. We compute daily returns, rolling volatility, cumulative returns, maximum drawdown, and identify trend regimes. The goal is to introduce quantitative analysts and data scientists to foundational equity time-series analysis.

## Learning Objectives

- Generate synthetic OHLCV data with realistic price dynamics
- Compute log returns and analyze their distributional properties
- Measure rolling volatility and understand volatility clustering
- Calculate maximum drawdown and drawdown duration
- Identify bull/bear market regimes from price data
- Analyze volume patterns around market events

## Business or Research Problem

A quantitative analyst needs to understand a stock's risk characteristics before constructing a portfolio or building a forecasting model. Key questions:
- What is the typical annualized volatility in each market regime?
- What was the maximum drawdown during the crash period?
- Is there volatility clustering (GARCH effects)?
- How quickly did the price recover from the crash?

## Why This Analysis Matters

- **Risk management**: Understanding volatility regimes helps set appropriate stop-loss levels and position sizes
- **Derivatives pricing**: Options pricing depends critically on realized volatility estimates
- **Portfolio construction**: Drawdown analysis informs maximum acceptable loss thresholds
- **Regime-switching models**: Identifying bull/bear regimes is foundational for factor model design
- **Stress testing**: Knowing crash dynamics helps stress test portfolio models

## Dataset Overview

**5 years of daily data** (1,260 trading days, 2019–2023).

| Column | Description |
|---|---|
| `date` | Trading date |
| `open` | Opening price |
| `high` | Intraday high |
| `low` | Intraday low |
| `close` | Closing price |
| `volume` | Shares traded |

Derived columns: `daily_return`, `rolling_volatility_7`, `rolling_volatility_30`, `cumulative_return`, `drawdown`.

Simulated regimes: Bull (Year 1–2), Crash (~Year 2.5), Recovery (Year 3–5).

## Dataset Source and License Notes

Fully synthetic dataset generated with NumPy geometric Brownian motion (GBM) simulation with regime-switching drift parameters. No real stock prices are used. Educational use only.

## Environment Setup

```bash
pip install pandas numpy matplotlib seaborn scipy
```

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='tab10')
print('Libraries loaded.')

## Configuration / Constants

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

S0          = 100.0   # starting price
TRADING_DAYS = 1260   # ~5 years

# Regime parameters (mu=daily drift, sigma=daily vol)
REGIMES = [
    {'name': 'Bull',     'days': 500, 'mu':  0.0006, 'sigma': 0.012},
    {'name': 'Crash',    'days': 120, 'mu': -0.005,  'sigma': 0.030},
    {'name': 'Recovery', 'days': 640, 'mu':  0.0008, 'sigma': 0.015},
]

ROLLING_SHORT = 7
ROLLING_LONG  = 30

print('Regime schedule:')
for r in REGIMES:
    print(f"  {r['name']}: {r['days']} days, mu={r['mu']}, sigma={r['sigma']}")

## Dataset Download or Loading

We simulate price paths using Geometric Brownian Motion (GBM) with regime-switching parameters. GBM models the log-normal price distribution used as the basis for Black-Scholes options pricing.

In [ ]:
# Generate returns by regime
returns_list = []
for r in REGIMES:
    daily_returns = r['mu'] + r['sigma'] * np.random.randn(r['days'])
    returns_list.append(daily_returns)

all_returns = np.concatenate(returns_list)

# Cumulative price path
prices = S0 * np.exp(np.cumsum(all_returns))
prices = np.concatenate([[S0], prices])[: TRADING_DAYS]

# Trading date index (skip weekends)
dates = pd.bdate_range(start='2019-01-02', periods=TRADING_DAYS)

# OHLCV simulation
close = prices
daily_range = close * np.random.uniform(0.005, 0.025, TRADING_DAYS)
high  = close + daily_range * 0.6
low   = close - daily_range * 0.4
open_ = close - (close - close * (1 + np.random.uniform(-0.01, 0.01, TRADING_DAYS)))
volume = np.random.randint(500_000, 3_000_000, TRADING_DAYS)

df = pd.DataFrame({
    'date'  : dates,
    'open'  : open_.round(2),
    'high'  : high.round(2),
    'low'   : low.round(2),
    'close' : close.round(2),
    'volume': volume
})
df.set_index('date', inplace=True)

# Derived columns
df['daily_return']         = df['close'].pct_change()
df['log_return']           = np.log(df['close'] / df['close'].shift(1))
df['rolling_vol_7']        = df['daily_return'].rolling(ROLLING_SHORT).std() * np.sqrt(252)
df['rolling_vol_30']       = df['daily_return'].rolling(ROLLING_LONG).std() * np.sqrt(252)
df['cumulative_return']    = (1 + df['daily_return']).cumprod() - 1

# Drawdown calculation
rolling_max = df['close'].cummax()
df['drawdown'] = (df['close'] - rolling_max) / rolling_max

print(f'Shape: {df.shape}')
df.head()

## Data Validation Checks

In [ ]:
print('=== Validation ===')
print(f'Shape          : {df.shape}')
print(f'Missing values :\n{df.isnull().sum()}')
print(f'Price range    : ${df["close"].min():.2f} – ${df["close"].max():.2f}')
print(f'Max drawdown   : {df["drawdown"].min():.2%}')
assert (df['high'] >= df['close']).all()
assert (df['low']  <= df['close']).all()
print('\nAll checks passed.')

## Data Cleaning

The first row has NaN for returns (no prior close). We drop it for return-based analyses.

In [ ]:
df_ret = df.dropna(subset=['daily_return'])
print(f'Rows after dropping NaN returns: {len(df_ret)}')
df_ret[['close', 'daily_return', 'rolling_vol_30', 'drawdown']].describe().round(4)

## Exploratory Data Analysis

Plot the full price history with regime annotations.

In [ ]:
# Compute regime boundaries
bull_end = dates[REGIMES[0]['days']]
crash_end = dates[REGIMES[0]['days'] + REGIMES[1]['days']]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['close'], color='steelblue', linewidth=1.5, label='Close Price')
ax.axvspan(dates[0], bull_end, alpha=0.08, color='green', label='Bull Regime')
ax.axvspan(bull_end, crash_end, alpha=0.15, color='red', label='Crash Regime')
ax.axvspan(crash_end, dates[-1], alpha=0.08, color='gold', label='Recovery Regime')
ax.set_title('Simulated Stock Price with Market Regimes', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

**Interpretation:** The price chart clearly shows three distinct regimes — a bull market with steady appreciation, a sharp crash (drawdown period), and a subsequent recovery. This structure is common in equity markets and is important to account for in any return analysis.

## Univariate Analysis

Distribution of daily returns and comparison to a normal distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

returns = df_ret['daily_return'].dropna()
x_range = np.linspace(returns.min(), returns.max(), 200)
normal_fit = stats.norm.pdf(x_range, returns.mean(), returns.std())

axes[0].hist(returns, bins=60, density=True, color='steelblue', edgecolor='white', alpha=0.7, label='Actual Returns')
axes[0].plot(x_range, normal_fit, color='red', linewidth=2, label='Normal Fit')
axes[0].set_title('Return Distribution vs Normal')
axes[0].set_xlabel('Daily Return')
axes[0].legend()

stats.probplot(returns, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Daily Returns')

plt.tight_layout()
plt.show()

print(f'Mean return   : {returns.mean():.4f}')
print(f'Std dev       : {returns.std():.4f}')
print(f'Skewness      : {returns.skew():.4f}')
print(f'Kurtosis      : {returns.kurtosis():.4f}  (excess; normal=0)')

**Interpretation:** The return distribution shows fat tails (excess kurtosis > 0) compared to the normal distribution — a well-known property of real financial returns called leptokurtosis. The Q-Q plot confirms the tails deviate from normality, which has important implications for risk models that assume Gaussian returns.

## Bivariate / Multivariate Analysis

Rolling volatility comparison and drawdown curve.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

axes[0].plot(df.index, df['close'], color='steelblue', linewidth=1.2)
axes[0].set_title('Close Price', fontsize=12)
axes[0].set_ylabel('Price (USD)')

axes[1].plot(df.index, df['rolling_vol_7'],  color='orange', label='7-Day Annualized Vol', linewidth=1)
axes[1].plot(df.index, df['rolling_vol_30'], color='crimson', label='30-Day Annualized Vol', linewidth=1.5)
axes[1].set_title('Rolling Annualized Volatility', fontsize=12)
axes[1].set_ylabel('Volatility')
axes[1].legend()

axes[2].fill_between(df.index, df['drawdown'], 0, color='red', alpha=0.5)
axes[2].set_title('Drawdown from Peak', fontsize=12)
axes[2].set_ylabel('Drawdown')
axes[2].set_xlabel('Date')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.show()

**Interpretation:** The three-panel chart illustrates the relationship between price, volatility, and drawdown. During the crash period, rolling volatility spikes dramatically — a phenomenon called volatility clustering. The drawdown chart shows the depth and duration of the crash, and confirms the recovery as drawdown returns toward zero.

## Feature-Specific Insights

Volume patterns around the crash and regime-level return statistics.

In [ ]:
# Assign regime label
regime_labels = (['Bull'] * REGIMES[0]['days'] +
                 ['Crash'] * REGIMES[1]['days'] +
                 ['Recovery'] * REGIMES[2]['days'])
regime_labels = regime_labels[:TRADING_DAYS]
df['regime'] = regime_labels

regime_stats = df_ret.join(df[['regime']]).groupby('regime').agg(
    avg_return=('daily_return', 'mean'),
    vol=('daily_return', 'std'),
    count=('daily_return', 'count')
).round(5)
regime_stats['annualized_vol'] = (regime_stats['vol'] * np.sqrt(252)).round(4)
regime_stats['annualized_return'] = (regime_stats['avg_return'] * 252).round(4)
print(regime_stats)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
colors = df['regime'].map({'Bull': 'green', 'Crash': 'red', 'Recovery': 'goldenrod'})
ax.bar(df.index, df['volume'] / 1e6, color=colors, alpha=0.6, width=1)
ax.set_title('Daily Volume by Regime (Million Shares)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Volume (M)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='green', label='Bull'), Patch(color='red', label='Crash'), Patch(color='goldenrod', label='Recovery')])
plt.tight_layout()
plt.show()

**Interpretation:** The crash regime shows high and erratic daily returns with dramatically elevated annualized volatility. Volume during crash periods tends to be higher on average, consistent with forced selling and panic behavior in real markets.

## Statistical Checks

Test for fat tails using Jarque-Bera normality test and compute Value-at-Risk (VaR).

In [ ]:
# Jarque-Bera test
jb_stat, jb_pval = stats.jarque_bera(returns.dropna())
print(f'Jarque-Bera test: stat={jb_stat:.2f}, p={jb_pval:.4e}')
print(f'Fat tails: {"Yes (reject normality)" if jb_pval < 0.05 else "No"}')

# Value at Risk (95% and 99%)
var_95 = np.percentile(returns.dropna(), 5)
var_99 = np.percentile(returns.dropna(), 1)
print(f'\nHistorical VaR (95%): {var_95:.4f} ({var_95*100:.2f}%)')
print(f'Historical VaR (99%): {var_99:.4f} ({var_99*100:.2f}%)')

# Max drawdown
max_dd = df['drawdown'].min()
max_dd_date = df['drawdown'].idxmin()
print(f'\nMax Drawdown: {max_dd:.2%} on {max_dd_date.date()}')

## Visual Analysis

Yearly return boxplots and autocorrelation of squared returns (volatility clustering evidence).

In [ ]:
df_ret2 = df_ret.copy()
df_ret2['year'] = df_ret2.index.year

fig, ax = plt.subplots(figsize=(11, 5))
df_ret2.boxplot(column='daily_return', by='year', ax=ax,
                boxprops=dict(color='steelblue'),
                medianprops=dict(color='red', linewidth=2))
ax.set_title('Daily Returns by Year')
ax.set_xlabel('Year')
ax.set_ylabel('Daily Return')
plt.suptitle('')
plt.tight_layout()
plt.show()

**Interpretation:** The return boxplot by year shows the crash year has a much wider spread (high variance) and a negative median return. Recovery years show positive median returns with narrowing variance, confirming regime stabilization.

## Key Findings

In [ ]:
total_return = df['close'].iloc[-1] / df['close'].iloc[0] - 1
ann_vol_full = returns.std() * np.sqrt(252)

print('=== Key Findings ===')
print(f'1. Total 5-year return      : {total_return:.2%}')
print(f'2. Full-period annualized vol: {ann_vol_full:.2%}')
print(f'3. Crash regime ann. vol    : {regime_stats.loc["Crash", "annualized_vol"]:.2%}')
print(f'4. Bull regime ann. vol     : {regime_stats.loc["Bull", "annualized_vol"]:.2%}')
print(f'5. Max drawdown             : {max_dd:.2%} on {max_dd_date.date()}')
print(f'6. 95% Historical VaR       : {var_95:.3%} per day')
print(f'7. 99% Historical VaR       : {var_99:.3%} per day')
print(f'8. Fat tails (JB test)      : p={jb_pval:.2e} (reject normality)')

## Limitations

1. **GBM simplification**: Real stock prices exhibit jumps, mean-reversion, and GARCH effects not fully captured by simple GBM.
2. **No market microstructure**: Bid-ask spread, intraday patterns, and order book dynamics are absent.
3. **Single asset**: Portfolio-level analysis requires correlation matrices across multiple assets.
4. **No dividends or splits**: Corporate actions significantly affect return calculations in real data.
5. **Regime boundaries are fixed**: Real regimes are not known in advance and require hidden Markov models or change-point detection.

## Recommendations / Next Steps

1. Fit a GARCH(1,1) model to the returns to quantify volatility persistence.
2. Implement a Hidden Markov Model (HMM) to detect regime changes without knowing boundaries in advance.
3. Build a portfolio of 5–10 such simulated assets and compute correlation-adjusted risk metrics.
4. Compare Historical VaR to Parametric VaR and CVaR (Expected Shortfall).
5. Backtest a simple volatility-targeting strategy (scale position size inversely with rolling volatility).

## Common Mistakes

1. Assuming normally distributed returns when computing VaR — leads to systematic underestimation of tail risk.
2. Computing annualized volatility without adjusting for the number of trading days (use 252, not 365).
3. Ignoring regime shifts when building forecasting models — parameters estimated in a bull market do not transfer to a crash regime.
4. Using price-level instead of returns for correlation analysis.
5. Confusing total drawdown depth with drawdown duration — both matter for risk assessment.

## Mini Challenge / Exercises

1. Increase the crash regime severity (mu=-0.010) and measure the new maximum drawdown.
2. Calculate the Sharpe ratio for each regime: (annualized return - 0.03) / annualized volatility.
3. Compute a 252-day rolling Sharpe ratio and plot it over time.
4. Identify the longest consecutive streak of negative daily returns.
5. Build a simple mean-reversion signal: buy when price drops more than 2 rolling standard deviations in 30 days.

## Final Summary / Key Takeaways

- Stock returns exhibit **fat tails and volatility clustering** — properties that violate simple normality assumptions used in basic risk models.
- **Rolling volatility** is a practical tool for monitoring regime changes in real time; spikes in 7-day vol precede or accompany crash periods.
- **Maximum drawdown** is a critical risk metric for investors and portfolio managers — it captures the worst peak-to-trough experience.
- **Regime identification** is essential before any return-based modeling — parameters estimated in one regime will not generalize to another.
- **Historical VaR** provides a useful tail-risk estimate but should be supplemented with Expected Shortfall (CVaR) for regulatory and stress-testing purposes.